# Training Keyword Spotting
This notebook builds on the Colab in which we used the pre-trained [micro_speech](https://github.com/tensorflow/tensorflow/tree/v2.4.1/tensorflow/lite/micro/examples/micro_speech) example.

**It is now time for you to train your own custom keyword spotting model using the Speech Commands dataset!**

#Setup

## Import packages
Clone the TensorFlow Github Repository, which contains the relevant code required to run this assignment.

In [ ]:
# %pip install "setuptools>=65.0.0,<81"
# %pip install "pip==25.0"
# %pip install "tf_keras==2.18.0"
%pip uninstall -y tensorflow-metal
import subprocess
from importlib.metadata import version, PackageNotFoundError
import tensorflow as tf
import numpy as np

# Desired versions

tf_version = "2.18.0"
hub_version = "0.16.1"
datasets_version = "4.9.9"
numpy_version = "1.26.4"
protobuf_version = "5.29.5"
tfmot_opt_version = "0.8.0"


# Try importing optional packages
# try:
#     import tensorflow_hub as hub
# except ImportError:
#     hub = None
#
# try:
#     import tensorflow_datasets as tfds
# except ImportError:
#     tfds = None
#
# # Version checks
# version_mismatch = (
#     tf.__version__ != tf_version or
#     (hub and hub.__version__ != hub_version) or
#     (tfds and tfds.__version__ != datasets_version) or
#     not np.__version__.startswith(numpy_version)
# )
#
# if version_mismatch:
#     print(f"TensorFlow: {tf.__version__} → {tf_version}")
#     if hub: print(f"TensorFlow Hub: {hub.__version__} → {hub_version}")
#     if tfds: print(f"TensorFlow Datasets: {tfds.__version__} → {datasets_version}")
#     print(f"NumPy: {np.__version__} → {numpy_version}")
#
#     %pip --no-input uninstall "tensorflow" "tensorflow_hub" "tensorflow_datasets" "numpy" "protobuf"
#
#     %pip --no-input install "tensorflow>={tf_version}"
#     %pip --no-input install "tensorflow_hub>={hub_version}"
#     %pip --no-input install "tensorflow_datasets>={datasets_version}"
#     %pip --no-input install "numpy>={numpy_version}"
#     %pip --no-input install "protobuf>={protobuf_version}"
#
#     print("\nRequired versions installed.")
#     print("Please restart the runtime: Runtime > Restart session, then run all cells again.")
# else:
#     print("All base packages are already at the specified versions.")
#
# # TensorFlow Model Optimization version check
# try:
#     tfmot_installed_version = version("tensorflow_model_optimization")
#     if tfmot_installed_version != tfmot_opt_version:
#         print(f"TF MOT: {tfmot_installed_version} → {tfmot_opt_version}")
#         %pip --no-input uninstall "tensorflow-model-optimization"
#         %pip --no-input install "tensorflow-model-optimization>={tfmot_opt_version}"
#     else:
#         print("tensorflow-model-optimization is already at the correct version.")
# except PackageNotFoundError:
#     print("tensorflow-model-optimization not found. Installing now.")
#     %pip --no-input install "tensorflow-model-optimization=={tfmot_opt_version}"


In [ ]:
# !wget https://github.com/tensorflow/tensorflow/archive/v2.4.1.zip
# !unzip v2.4.1.zip &> 0
# !mv tensorflow-2.4.1/ tensorflow/

In [1]:
import tensorflow.compat.v1 as tf
#assert(tf.VERSION == "1.15.0") #Make sure you install Tensorflow 1.15.0 and restart the runtime or this will fail!
# Now that the runtime is set import things!
import sys
# We add this path so we can import the speech processing modules.
sys.path.append("./tensorflow/tensorflow/examples/speech_commands/")
import input_data
import models
import numpy as np
import pickle

## Configure Your Model!
Select your keywords and model settings with which to train!

**This is where you need to make choices and input data!**

```WANTED_WORDS``` = A comma-delimited string of the words you want to train for (e.g., "yes,no"). All the other words you do not select will be used to train an "unknown" label so that the model does not just recognize speech but your specific words. Audio data with no spoken words will be used to train a "silence" label. We suggest picking 2-4 words for best results.

Options for target words are (PICK FROM THIS LIST FOR BEST RESULTS): "one", "two", "three", "four", "five", "six", "seven", "eight", "nine",  "yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go", “backward”, “forward”, “follow”, “learn”,

Additional words that will be used to help train the "unkown" label are: "bed", "bird", "cat", "dog", "happy", "house", "marvin", "sheila", "tree", "wow"

In [2]:
#WANTED_WORDS = "on,off,stop,go,dog,wow,cat,happy"
WANTED_WORDS = "rot,grün,blau,gelb,pink,türkis,weiss,aus"

The number of training steps and learning rates can be specified as comma-separated strings to define the amount/rate at each stage. For example, ```TRAINING_STEPS="12000,3000"``` and ```LEARNING_RATE="0.001,0.0001"``` will run 12,000 training steps with a rate of 0.001 followed by 3,000 final steps with a learning rate of 0.0001. These are good default values to work off of when you choose your values as the course staff has gotten this to work well with those values in the past!

In [3]:
TRAINING_STEPS =  "12000,3000"
LEARNING_RATE = "0.001,0.0001"

We suggest you leave the ```MODEL_ARCHITECTURE``` as tiny_conv the first time but if you would like to do this again and explore additional models some options are: ```single_fc, conv, low_latency_conv, low_latency_svdf, tiny_embedding_conv```

In [4]:
MODEL_ARCHITECTURE = 'tiny_conv'

In [5]:
# Calculate the total number of steps, which is used to identify the checkpoint
# file name.
TOTAL_STEPS = str(sum(map(lambda string: int(string), TRAINING_STEPS.split(","))))

# Print the configuration to confirm it
print("Training these words: %s" % WANTED_WORDS)
print("Training steps in each stage: %s" % TRAINING_STEPS)
print("Learning rate in each stage: %s" % LEARNING_RATE)
print("Total number of training steps: %s" % TOTAL_STEPS)

Training these words: up,down,left,right
Training steps in each stage: 12000,3000
Learning rate in each stage: 0.001,0.0001
Total number of training steps: 15000


**DO NOT MODIFY** the following constants as they include filepaths used in this notebook and data that is shared during training and inference.

In [6]:
# Calculate the percentage of 'silence' and 'unknown' training samples required
# to ensure that we have equal number of samples for each label.
number_of_labels = WANTED_WORDS.count(',') + 1
number_of_total_labels = number_of_labels + 2 # for 'silence' and 'unknown' label
equal_percentage_of_training_samples = int(100.0/(number_of_total_labels))
SILENT_PERCENTAGE = equal_percentage_of_training_samples
UNKNOWN_PERCENTAGE = equal_percentage_of_training_samples

# Constants which are shared during training and inference
PREPROCESS = 'micro'
WINDOW_STRIDE = 20

# Constants used during training only
VERBOSITY = 'DEBUG'
EVAL_STEP_INTERVAL = '1000'
SAVE_STEP_INTERVAL = '1000'

# Constants for training directories and filepaths
DATASET_DIR =  'dataset/'
LOGS_DIR = 'logs/'
TRAIN_DIR = 'train/' # for training checkpoints and other files.

# Constants for inference directories and filepaths
import os
MODELS_DIR = 'models'
if not os.path.exists(MODELS_DIR):
  os.mkdir(MODELS_DIR)
MODEL_TF = os.path.join(MODELS_DIR, 'model.pb')
MODEL_TFLITE = os.path.join(MODELS_DIR, 'model.tflite')
FLOAT_MODEL_TFLITE = os.path.join(MODELS_DIR, 'float_model.tflite')
MODEL_TFLITE_MICRO = os.path.join(MODELS_DIR, 'model.cc')
SAVED_MODEL = os.path.join(MODELS_DIR, 'saved_model')

# Constants for Quantization
QUANT_INPUT_MIN = 0.0
QUANT_INPUT_MAX = 26.0
QUANT_INPUT_RANGE = QUANT_INPUT_MAX - QUANT_INPUT_MIN

# Constants for audio process during Quantization and Evaluation
SAMPLE_RATE = 16000
CLIP_DURATION_MS = 1000
WINDOW_SIZE_MS = 30.0
FEATURE_BIN_COUNT = 40
BACKGROUND_FREQUENCY = 0.8
BACKGROUND_VOLUME_RANGE = 0.1
TIME_SHIFT_MS = 100.0

# URL for the dataset and train/val/test split
DATA_URL = ''
VALIDATION_PERCENTAGE = 10
TESTING_PERCENTAGE = 10

In [7]:
# Calculate the correct flattened input data shape for later use in model conversion
# since the model takes a flattened version of the spectrogram. The shape is number of
# overlapping windows times the number of frequency bins. For the default settings we have
# 40 bins (as set above) times 49 windows (as calculated below) so the shape is (1,1960)
def window_counter(total_samples, window_size, stride):
  '''helper function to count the number of full-length overlapping windows'''
  window_count = 0
  sample_index = 0
  while True:
    window = range(sample_index,sample_index+stride)
    if window.stop < total_samples:
      window_count += 1
    else:
      break

    sample_index += stride
  return window_count

OVERLAPPING_WINDOWS = window_counter(CLIP_DURATION_MS, int(WINDOW_SIZE_MS), WINDOW_STRIDE)
FLATTENED_SPECTROGRAM_SHAPE = (1, OVERLAPPING_WINDOWS * FEATURE_BIN_COUNT)

# Train the model

### Load in TensorBoard to visualize the training process.

As training progresses you should see the training status show up in the Tensorboard area. If this works it is very helpful for analyzing your training progress. Unfortunately, the staff has found that it sometimes doesn't start showing data for a while (~15 minutes) and sometimes doesn't show data until training completes (and instead shows ```No dashboards are active for the current data set```.). If it is working and then stops updating look to the top of the cell and click reconnect.

In [ ]:
# %pip install tensorboard

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir {LOGS_DIR}

### Launch Training

**ARE YOU USING A GPU INSTANCE!?!?!**
Make sure to check ```runtime->change runtime type``` as a GPU runtime will take ~2 hours to complete training whereas a CPU runtime with take ~10 hours!

**Importantly, every hour or so you should make sure to remind colab you are still there by clicking around in a cell or clicking reconnect on the tensorboard cell (after 90 minutes of no activity Colab may kill the instance)!** But/and beyond that you can absolutely minimize the window and continue with other work.

**If you run training a second time** we suggest restarting your runtime ```runtime->restart runtime``` in order to ensure all caches are cleared. However, do note that doing so will clear all cells and so you will have to run the entire file again.

If you would like to get more information on the training script you can find the source code for the script [here](https://github.com/tensorflow/tensorflow/blob/master/tensorflow/examples/speech_commands/train.py). In short it sets up the optimizer and preprocessor based on all of the flags we pass in!

Finally, by setting the ```VERBOSITY = 'DEBUG'``` above be aware that the training cell will print A LOT of information. Specifically you will get the accuracy and loss at each step as well as a confusion matrix every 1000 steps. We hope that is helpful in case TensorBoard fails to work. If you would like to run with less printouts you can change the setting to ```WARN``` or ```FATAL```. You will find this in the "Configure Your Model!" section.

In [ ]:
# %pip install tensorflow-metal==1.2.0


In [ ]:
import tensorflow as tf
tf.config.list_physical_devices()

In [8]:
%run tensorflow/tensorflow/examples/speech_commands/train.py --data_dir={DATASET_DIR} --wanted_words={WANTED_WORDS} --silence_percentage={SILENT_PERCENTAGE} --unknown_percentage={UNKNOWN_PERCENTAGE} --preprocess={PREPROCESS} --window_stride={WINDOW_STRIDE} --model_architecture={MODEL_ARCHITECTURE} --how_many_training_steps={TRAINING_STEPS} --learning_rate={LEARNING_RATE} --train_dir={TRAIN_DIR} --summaries_dir={LOGS_DIR} --verbosity={VERBOSITY} --eval_step_interval={EVAL_STEP_INTERVAL} --save_step_interval={SAVE_STEP_INTERVAL} --start_checkpoint=train/tiny_conv.ckpt-15000


INFO:tensorflow:Restoring parameters from train/tiny_conv.ckpt-15000


I0000 00:00:1772295733.626282  177295 mlir_graph_optimization_pass.cc:401] MLIR V1 optimization pass is not enabled
I0228 17:22:13.762009 8510337152 saver.py:1417] Restoring parameters from train/tiny_conv.ckpt-15000


INFO:tensorflow:Training from step: 15000 


I0228 17:22:13.767991 8510337152 train.py:213] Training from step: 15000 


DEBUG:tensorflow:Step #15000: rate 0.000100, accuracy 80.0%, cross entropy 0.496415


...

INFO:tensorflow:set_size=2165


I0228 17:22:16.101505 8510337152 train.py:302] set_size=2165


 [[251   1   8   1   1   1]
 [  2 133  31  41  26  30]
 [  3  21 372   8  15   6]
 [  2  26  16 355   7   0]
 [  1  13  16   6 370   6]
 [  2  12   9   3  12 358]]


W0228 17:22:18.583182 8510337152 train.py:321] Confusion Matrix:
 [[251   1   8   1   1   1]
 [  2 133  31  41  26  30]
 [  3  21 372   8  15   6]
 [  2  26  16 355   7   0]
 [  1  13  16   6 370   6]
 [  2  12   9   3  12 358]]


W0228 17:22:18.584005 8510337152 train.py:322] Final test accuracy = 84.9% (N=2165)


# Generating your Model
Just like with the pre-trained model we will now take the final checkpoint and convert it into a quantized TensorFlow Lite model.

### Generate a TensorFlow Model for Inference

Combine relevant training results (graph, weights, etc) into a single file for inference. This process is known as freezing a model and the resulting model is known as a frozen model/graph, as it cannot be further re-trained after this process.

In [14]:
# These commands modify the legacy TensorFlow example code to make it compatible with TensorFlow 2.x by replacing deprecated imports and removing
# runtime placeholders with constants, ensuring the model can be frozen and converted to TFLite without requiring dynamic inputs.

# Patch freeze.py to use the TF 2.x-compatible version of convert_variables_to_constants from tf.compat.v1.
!sed -i '' 's/from tensorflow\.python\.framework import graph_util/from tensorflow.compat.v1.graph_util import convert_variables_to_constants/' tensorflow/tensorflow/examples/speech_commands/freeze.py

# Update function call to match the new import (remove graph_util prefix).
!sed -i '' 's/graph_util\.convert_variables_to_constants/convert_variables_to_constants/' tensorflow/tensorflow/examples/speech_commands/freeze.py

# Replace placeholder for background_volume with a constant to eliminate runtime input requirements.
!sed -i '' "s/tf\.compat\.v1\.placeholder(tf\.float32, name='background_volume')/tf.constant(0.0, dtype=tf.float32, name='background_volume')/" tensorflow/tensorflow/examples/speech_commands/input_data.py

# Replace placeholder for time_shift_padding with a constant to avoid dynamic input during inference.
!sed -i  '' "s/tf\.compat\.v1\.placeholder(tf\.int32, (shape=)?\[2,2\], name='time_shift_padding')/tf.constant([[0, 0], [0, 0]], dtype=tf.int32, name='time_shift_padding')/" tensorflow/tensorflow/examples/speech_commands/input_data.py

# Replace placeholder for foreground_volume with a constant to ensure static graph for TFLite conversion.
!sed -i '' "s/tf\.compat\.v1\.placeholder(tf\.float32, name='foreground_volume')/tf.constant(1.0, dtype=tf.float32, name='foreground_volume')/" tensorflow/tensorflow/examples/speech_commands/input_data.py


In [9]:
#!rm -rf {SAVED_MODEL}
%run tensorflow/tensorflow/examples/speech_commands/freeze.py \
--wanted_words=$WANTED_WORDS \
--window_stride_ms=$WINDOW_STRIDE \
--preprocess=$PREPROCESS \
--model_architecture=$MODEL_ARCHITECTURE \
--start_checkpoint=$TRAIN_DIR$MODEL_ARCHITECTURE'.ckpt-'{TOTAL_STEPS} \
--save_format=saved_model \
--output_file={SAVED_MODEL}

ERROR:tensorflow:An interactive session is already active. This can cause out-of-memory errors or some other unexpected errors (due to the unpredictable timing of garbage collection) in some cases. You must explicitly call `InteractiveSession.close()` to release resources held by the other session(s). Please use `tf.Session()` if you intend to productionize.


E0228 17:47:10.392381 8510337152 session.py:1798] An interactive session is already active. This can cause out-of-memory errors or some other unexpected errors (due to the unpredictable timing of garbage collection) in some cases. You must explicitly call `InteractiveSession.close()` to release resources held by the other session(s). Please use `tf.Session()` if you intend to productionize.


ValueError: Variable first_weights already exists, disallowed. Did you mean to set reuse=True or reuse=tf.AUTO_REUSE in VarScope?

In [11]:
float_converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL)
float_tflite_model = float_converter.convert()

with open(FLOAT_MODEL_TFLITE, "wb") as f:
    f.write(float_tflite_model)

W0228 17:24:45.492806 8510337152 lite.py:2230] Invoking the TF1 implementation of TFLiteConverter because eager is disabled. Consider enabling eager.


Instructions for updating:
Use `tf.saved_model.load` instead.


W0228 17:24:45.493989 8510337152 deprecation.py:50] From /Users/peter/PycharmProjects/TinyML2/.venv/lib/python3.12/site-packages/tensorflow/lite/python/convert_saved_model.py:42: load (from tensorflow.python.saved_model.loader_impl) is deprecated and will be removed in a future version.
Instructions for updating:
Use `tf.saved_model.load` instead.


INFO:tensorflow:Restoring parameters from models/saved_model/variables/variables


I0228 17:24:45.501180 8510337152 saver.py:1417] Restoring parameters from models/saved_model/variables/variables


...

I0228 17:24:45.556871 8510337152 saver.py:1417] Restoring parameters from models/saved_model/variables/variables


INFO:tensorflow:The given SavedModel MetaGraphDef contains SignatureDefs with the following keys: {'serving_default'}


I0228 17:24:45.561789 8510337152 convert_saved_model.py:60] The given SavedModel MetaGraphDef contains SignatureDefs with the following keys: {'serving_default'}


INFO:tensorflow:Restoring parameters from models/saved_model/variables/variables


I0228 17:24:45.568218 8510337152 saver.py:1417] Restoring parameters from models/saved_model/variables/variables
I0000 00:00:1772295885.573756  177295 devices.cc:76] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0 (Note: TensorFlow was not compiled with CUDA or ROCm support)
I0000 00:00:1772295885.573827  177295 single_machine.cc:361] Starting new session
I0000 00:00:1772295885.580619  177295 devices.cc:76] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0 (Note: TensorFlow was not compiled with CUDA or ROCm support)
I0000 00:00:1772295885.580689  177295 single_machine.cc:361] Starting new session
E0000 00:00:1772295885.581070  177295 grappler_item_builder.cc:675] Init node first_weights/Assign doesn't exist in graph
I0228 17:24:45.589528 8510337152 lite.py:2581] Using experimental converter: If you encountered a problem please file a bug. You can opt-out by setting experimental_new_converter=False
W0000 00:00:1772295885.590902  177295 tf_t

### Generate a TensorFlow Lite Model

Convert the frozen graph into a TensorFlow Lite model, which is fully quantized for use with embedded devices.

The following cell will also print the model size, which will be under 20 kilobytes.

We download the dataset to use as a representative dataset for more thoughtful post training quantization.

In [12]:
model_settings = models.prepare_model_settings(
    len(input_data.prepare_words_list(WANTED_WORDS.split(','))),
    SAMPLE_RATE, CLIP_DURATION_MS, WINDOW_SIZE_MS,
    WINDOW_STRIDE, FEATURE_BIN_COUNT, PREPROCESS)
audio_processor = input_data.AudioProcessor(
    DATA_URL, DATASET_DIR,
    SILENT_PERCENTAGE, UNKNOWN_PERCENTAGE,
    WANTED_WORDS.split(','), VALIDATION_PERCENTAGE,
    TESTING_PERCENTAGE, model_settings, LOGS_DIR)

In [13]:

with tf.compat.v1.Session() as sess:

    converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    def representative_dataset_gen():
        for i in range(100):
            data, _ = audio_processor.get_data(
                1, i, model_settings,
                BACKGROUND_FREQUENCY,
                BACKGROUND_VOLUME_RANGE,
                TIME_SHIFT_MS,
                'testing',
                sess
            )
            flattened = np.array(data.flatten(), dtype=np.float32).reshape(FLATTENED_SPECTROGRAM_SHAPE)
            yield [flattened]

    converter.representative_dataset = representative_dataset_gen


    quantized_tflite_model = converter.convert()

    # Save to file
    with open(MODEL_TFLITE, "wb") as f:
        f.write(quantized_tflite_model)

    print(f"Quantized model saved to {MODEL_TFLITE}, size: {len(quantized_tflite_model)} bytes")


W0228 17:25:40.965523 8510337152 lite.py:2230] Invoking the TF1 implementation of TFLiteConverter because eager is disabled. Consider enabling eager.


INFO:tensorflow:Restoring parameters from models/saved_model/variables/variables


I0228 17:25:40.971419 8510337152 saver.py:1417] Restoring parameters from models/saved_model/variables/variables


INFO:tensorflow:The given SavedModel MetaGraphDef contains SignatureDefs with the following keys: {'serving_default'}


I0228 17:25:40.976409 8510337152 convert_saved_model.py:60] The given SavedModel MetaGraphDef contains SignatureDefs with the following keys: {'serving_default'}


...

INFO:tensorflow:The given SavedModel MetaGraphDef contains SignatureDefs with the following keys: {'serving_default'}


I0228 17:25:41.003304 8510337152 convert_saved_model.py:60] The given SavedModel MetaGraphDef contains SignatureDefs with the following keys: {'serving_default'}


INFO:tensorflow:Restoring parameters from models/saved_model/variables/variables


I0228 17:25:41.010575 8510337152 saver.py:1417] Restoring parameters from models/saved_model/variables/variables
I0000 00:00:1772295941.015906  177295 devices.cc:76] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0 (Note: TensorFlow was not compiled with CUDA or ROCm support)
I0000 00:00:1772295941.015969  177295 single_machine.cc:361] Starting new session
I0000 00:00:1772295941.022844  177295 devices.cc:76] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0 (Note: TensorFlow was not compiled with CUDA or ROCm support)
I0000 00:00:1772295941.022884  177295 single_machine.cc:361] Starting new session
E0000 00:00:1772295941.023302  177295 grappler_item_builder.cc:675] Init node first_weights/Assign doesn't exist in graph
I0228 17:25:41.025169 8510337152 lite.py:2581] Using experimental converter: If you encountered a problem please file a bug. You can opt-out by setting experimental_new_converter=False
W0000 00:00:1772295941.025839  177295 tf_t

InvalidArgumentError: Graph execution error:

Detected at node 'data/time_shift_padding' defined at (most recent call last):
    File "<frozen runpy>", line 198, in _run_module_as_main
    File "<frozen runpy>", line 88, in _run_code
    File "/Users/peter/PycharmProjects/TinyML2/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    File "/Users/peter/PycharmProjects/TinyML2/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    File "/Users/peter/PycharmProjects/TinyML2/.venv/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start
    File "/Users/peter/PycharmProjects/TinyML2/.venv/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 211, in start
    File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/asyncio/base_events.py", line 645, in run_forever
    File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/asyncio/base_events.py", line 1999

### Testing the accuracy after Quantization

Verify that the model we've exported is still accurate, using the TF Lite Python API and our test set.

In [ ]:
# Helper function to run inference
def run_tflite_inference_testSet(tflite_model_path, model_type="Float"):
  #
  # Load test data
  #
  np.random.seed(0) # set random seed for reproducible test results.
  with tf.Session() as sess:
  # with tf.compat.v1.Session() as sess: #replaces the above line for use with TF2.x
    test_data, test_labels = audio_processor.get_data(
        -1, 0, model_settings, BACKGROUND_FREQUENCY, BACKGROUND_VOLUME_RANGE,
        TIME_SHIFT_MS, 'testing', sess)
  test_data = np.expand_dims(test_data, axis=1).astype(np.float32)

  #
  # Initialize the interpreter
  #
  interpreter = tf.lite.Interpreter(tflite_model_path)
  interpreter.allocate_tensors()
  input_details = interpreter.get_input_details()[0]
  output_details = interpreter.get_output_details()[0]

  #
  # For quantized models, manually quantize the input data from float to integer
  #
  if model_type == "Quantized":
    input_scale, input_zero_point = input_details["quantization"]
    test_data = test_data / input_scale + input_zero_point
    test_data = test_data.astype(input_details["dtype"])

  #
  # Evaluate the predictions
  #
  correct_predictions = 0
  for i in range(len(test_data)):
    interpreter.set_tensor(input_details["index"], test_data[i])
    interpreter.invoke()
    output = interpreter.get_tensor(output_details["index"])[0]
    top_prediction = output.argmax()
    correct_predictions += (top_prediction == test_labels[i])

  print('%s model accuracy is %f%% (Number of test samples=%d)' % (
      model_type, (correct_predictions * 100) / len(test_data), len(test_data)))

In [ ]:
# Compute float model accuracy
run_tflite_inference_testSet(FLOAT_MODEL_TFLITE)

# Compute quantized model accuracy
run_tflite_inference_testSet(MODEL_TFLITE, model_type='Quantized')

# Testing the model with your own data!
Now comes the fun part. It's time to test your model with your own realworld data. We'll proceed in the same way we tested the pre-trained model. Have fun!

### Importing packages

In [ ]:
!pip install ffmpeg-python &> 0
from IPython.display import HTML, Audio
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
from scipy.io.wavfile import read as wav_read
import io
import ffmpeg
!pip install librosa
import librosa
import scipy.io.wavfile
!git clone https://github.com/petewarden/extract_loudest_section.git
!make -C extract_loudest_section/
print("Packages Imported, Extract_Loudest_Section Built")

### Define the helper function to run inference

In [ ]:
# Helper function to run inference (on a single input this time)
# Note: this also includes additional manual pre-processing
TF_SESS = tf.compat.v1.InteractiveSession()
def run_tflite_inference_singleFile(tflite_model_path, custom_audio, sr_custom_audio, model_type="Float"):
  #
  # Preprocess the sample to get the features we pass to the model
  #
  # First re-sample to the needed rate (and convert to mono if needed)
  custom_audio_resampled = librosa.resample(y = librosa.to_mono(np.float64(custom_audio)), orig_sr = sr_custom_audio, target_sr = SAMPLE_RATE)
  # Then extract the loudest one second
  scipy.io.wavfile.write('custom_audio.wav', SAMPLE_RATE, np.int16(custom_audio_resampled))
  !/tmp/extract_loudest_section/gen/bin/extract_loudest_section custom_audio.wav ./trimmed
  # Finally pass it through the TFLiteMicro preprocessor to produce the
  # spectrogram/MFCC input that the model expects
  custom_model_settings = models.prepare_model_settings(
      0, SAMPLE_RATE, CLIP_DURATION_MS, WINDOW_SIZE_MS,
      WINDOW_STRIDE, FEATURE_BIN_COUNT, PREPROCESS)
  custom_audio_processor = input_data.AudioProcessor(None, None, 0, 0, '', 0, 0,
                                                    model_settings, None)
  custom_audio_preprocessed = custom_audio_processor.get_features_for_wav(
                                        'trimmed/custom_audio.wav', model_settings, TF_SESS)
  # Reshape the output into a 1,1960 matrix as that is what the model expects
  custom_audio_input = custom_audio_preprocessed[0].flatten()
  test_data = np.reshape(custom_audio_input,(1,len(custom_audio_input)))

  #
  # Initialize the interpreter
  #
  interpreter = tf.lite.Interpreter(tflite_model_path)
  interpreter.allocate_tensors()
  input_details = interpreter.get_input_details()[0]
  output_details = interpreter.get_output_details()[0]

  #
  # For quantized models, manually quantize the input data from float to integer
  #
  if model_type == "Quantized":
    input_scale, input_zero_point = input_details["quantization"]
    test_data = test_data / input_scale + input_zero_point
    test_data = test_data.astype(input_details["dtype"])

  #
  # Run the interpreter
  #
  interpreter.set_tensor(input_details["index"], test_data)
  interpreter.invoke()
  output = interpreter.get_tensor(output_details["index"])[0]
  top_prediction = output.argmax()

  #
  # Translate the output
  #
  top_prediction_str = ''
  if top_prediction >= 2:
    top_prediction_str = WANTED_WORDS.split(',')[top_prediction-2]
  elif top_prediction == 0:
    top_prediction_str = 'silence'
  else:
    top_prediction_str = 'unknown'

  print('%s model guessed the value to be %s' % (model_type, top_prediction_str))

### Define the audio importing function
Adapted from: https://ricardodeazambuja.com/deep_learning/2019/03/09/audio_and_video_google_colab/ and https://colab.research.google.com/drive/1Z6VIRZ_sX314hyev3Gm5gBqvm1wQVo-a#scrollTo=RtMcXr3o6gxN

In [ ]:
AUDIO_HTML = """
<script>
var my_div = document.createElement("DIV");
var my_p = document.createElement("P");
var my_btn = document.createElement("BUTTON");
var t = document.createTextNode("Press to start recording");

my_btn.appendChild(t);
//my_p.appendChild(my_btn);
my_div.appendChild(my_btn);
document.body.appendChild(my_div);

var base64data = 0;
var reader;
var recorder, gumStream;
var recordButton = my_btn;

var handleSuccess = function(stream) {
  gumStream = stream;
  var options = {
    //bitsPerSecond: 8000, //chrome seems to ignore, always 48k
    mimeType : 'audio/webm;codecs=opus'
    //mimeType : 'audio/webm;codecs=pcm'
  };
  //recorder = new MediaRecorder(stream, options);
  recorder = new MediaRecorder(stream);
  recorder.ondataavailable = function(e) {
    var url = URL.createObjectURL(e.data);
    var preview = document.createElement('audio');
    preview.controls = true;
    preview.src = url;
    document.body.appendChild(preview);

    reader = new FileReader();
    reader.readAsDataURL(e.data);
    reader.onloadend = function() {
      base64data = reader.result;
      //console.log("Inside FileReader:" + base64data);
    }
  };
  recorder.start();
  };

recordButton.innerText = "Recording... press to stop";

navigator.mediaDevices.getUserMedia({audio: true}).then(handleSuccess);


function toggleRecording() {
  if (recorder && recorder.state == "recording") {
      recorder.stop();
      gumStream.getAudioTracks()[0].stop();
      recordButton.innerText = "Saving the recording... pls wait!"
  }
}

// https://stackoverflow.com/a/951057
function sleep(ms) {
  return new Promise(resolve => setTimeout(resolve, ms));
}

var data = new Promise(resolve=>{
//recordButton.addEventListener("click", toggleRecording);
recordButton.onclick = ()=>{
toggleRecording()

sleep(2000).then(() => {
  // wait 2000ms for the data to be available...
  // ideally this should use something like await...
  //console.log("Inside data:" + base64data)
  resolve(base64data.toString())

});

}
});

</script>
"""

def get_audio():
  display(HTML(AUDIO_HTML))
  data = eval_js("data")
  binary = b64decode(data.split(',')[1])

  process = (ffmpeg
    .input('pipe:0')
    .output('pipe:1', format='wav', ac='1')
    .run_async(pipe_stdin=True, pipe_stdout=True, pipe_stderr=True, quiet=True, overwrite_output=True)
  )
  output, err = process.communicate(input=binary)

  riff_chunk_size = len(output) - 8
  # Break up the chunk size into four bytes, held in b.
  q = riff_chunk_size
  b = []
  for i in range(4):
      q, r = divmod(q, 256)
      b.append(r)

  # Replace bytes 4:8 in proc.stdout with the actual size of the RIFF chunk.
  riff = output[:4] + bytes(b) + output[8:]

  sr, audio = wav_read(io.BytesIO(riff))

  return audio, sr
print("Chrome Audio Recorder Defined")

### Record your own audio and test the model!
After you run the record cell wait for the stop button to appear then start recording and then press the button to stop the recording once you have said the word!

In [ ]:
custom_audio, sr_custom_audio = get_audio()
print("DONE")

In [ ]:
# Then test the model
run_tflite_inference_singleFile(MODEL_TFLITE, custom_audio, sr_custom_audio, model_type="Quantized")